In [60]:
from jamo import h2j, j2hcj
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pandas as pd
import joblib

In [61]:
data = pd.read_csv('korean_loanwords.csv')

def preprocess_hangul(text):
    """Decomposes '학교' into 'ㅎㅏㄱㄱㅛ' for better feature extraction.
    Otherwise, different words with same onset consonant will be treated as completely different words."""
    return j2hcj(h2j(text))

def count_eu(word: str) -> int:
    """Count how many 'ㅡ' vowels appear in the original word."""
    return word.count("ㅡ")

def count_rieul_initial(word: str) -> int:
    """Return 1 if the word starts with ㄹ (initial consonant of first syllable), else 0."""
    if not word:
        return 0
    first = word[0]
    decomposed = h2j(first)
    if decomposed and decomposed[0] == "ㄹ":
        return 1
    return 0

data['word'] = data['word'].apply(preprocess_hangul)
data['num_eu'] = data['word'].apply(count_eu)
data['num_rieul_initial'] = data['word'].apply(count_rieul_initial)

print(len(data))

data.drop_duplicates(subset=['word'], inplace=True)

print(len(data))

count_label_0 = (data['label'] == 0).sum()
count_label_1 = (data['label'] == 1).sum()
print(f"Label 0 count: {count_label_0}")
print(f"Label 1 count: {count_label_1}")

X_train, X_test, y_train, y_test = train_test_split(data['word'], data['label'], test_size=0.2, random_state=42)

25254
5483
Label 0 count: 2716
Label 1 count: 2767


In [62]:
lr_model = Pipeline([
    ('vectorizer', CountVectorizer(analyzer='char', ngram_range=(1, 4))),
    ('classifier', LogisticRegression())
])

In [63]:
lr_model.fit(X_train, y_train)

,steps,"[('vectorizer', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [64]:
lr_model.score(X_test, y_test)

0.918869644484959

In [65]:
test_words = ["인터넷", "강아지", "피자", "구름", "소파"]
for word in test_words:
    processed = preprocess_hangul(word)
    prediction = lr_model.predict([processed])[0]
    prob = lr_model.predict_proba([processed])[0][1]
    
    origin = "Loanword" if prediction == 1 else "Native/Sino"
    print(f"Word: {word} | Prediction: {origin} ({prob:.2%} confidence)")

Word: 인터넷 | Prediction: Loanword (99.41% confidence)
Word: 강아지 | Prediction: Native/Sino (0.39% confidence)
Word: 피자 | Prediction: Loanword (88.87% confidence)
Word: 구름 | Prediction: Native/Sino (2.68% confidence)
Word: 소파 | Prediction: Native/Sino (49.10% confidence)


In [66]:
joblib.dump(lr_model, 'loanword_model.pkl')

['loanword_model.pkl']

In [68]:
s = "아이스크림"
print(lr_model.predict([j2hcj(h2j(s))])[0])
print(lr_model.predict_proba([j2hcj(h2j(s))])[0])

1
[0.00493522 0.99506478]


In [ ]:
# Test cases: (word, expected_label). 1 = loanword, 0 = native/Sino-Korean.
TEST_CASES = [
    # Clear loanwords (expect 1)
    ("인터넷", 1),
    ("피자", 1),
    ("컴퓨터", 1),
    ("아이스크림", 1),
    ("스마트폰", 1),
    ("커피", 1),
    ("샌드위치", 1),
    # Clear native / Sino-Korean (expect 0)
    ("강아지", 0),
    ("구름", 0),
    ("그림자", 0),
    ("물", 0),
    ("하늘", 0),
    ("학교", 0),
    ("친구", 0),
    ("사랑", 0),
    ("눈", 0),
    ("바람", 0),
    # Edge / often confused
    ("소파", 1),   # loanword
    ("라디오", 1), # loanword (starts with ㄹ)
    ("레몬", 1),   # loanword
]

print("word        | expected | predicted | confidence | pass")
print("-" * 60)
passed = 0
for word, expected in TEST_CASES:
    processed = preprocess_hangul(word)
    pred = lr_model.predict([processed])[0]
    prob = lr_model.predict_proba([processed])[0][1]
    ok = "✓" if pred == expected else "✗"
    if pred == expected:
        passed += 1
    print(f"{word:<11} | {expected}        | {pred}         | {prob:.2%}     | {ok}")

print("-" * 60)
print(f"Passed: {passed}/{len(TEST_CASES)}")